# 🍋 Citron — ANIMA LoRA Trainer v5 (Kaggle Edition)

Train LoRA adapters for the [Anima](https://huggingface.co/circlestone-labs/Anima) diffusion model using [kohya-ss/sd-scripts](https://github.com/kohya-ss/sd-scripts), adapted for **Kaggle Notebooks**.

---

## ⚙️ Before You Start

| Step | What to do |
|------|------------|
| 1 | In the right-hand **Notebook settings** panel → **Accelerator** → select **GPU T4 x2** or **P100** |
| 2 | Enable **Internet** in the same panel (required to download models) |
| 3 | Upload your dataset as a Kaggle Dataset **or** place a `.zip` file in the working directory |
| 4 | Run cells top to bottom |

## 📁 Kaggle Path Reference

| Purpose | Path |
|---------|------|
| Working directory (read/write, ~20 GB free) | `/kaggle/working/` |
| Attached Kaggle Datasets (read-only) | `/kaggle/input/<dataset-slug>/` |
| Output files (auto-saved on commit) | `/kaggle/working/` |

## ⚠️ Session Limit

Kaggle gives **~9 hours** of GPU time per session (free tier: 30 GPU-hours/week).  
The step estimator below will warn you if you risk running out of time.  
**Recommended maximum: ~2000 steps** on a T4 for a safe single-session run.

---

## Dataset Format

Use a **flat directory** of image + caption pairs:

```
my_dataset/
  image001.png
  image001.txt   ← comma-separated tags
  image002.jpg
  image002.txt
```

Caption files must share the same base name as their image and use `.txt`.

## Cell 1 — GPU Check

In [ ]:
import subprocess, sys, os

def _run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip()

gpu_info = _run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
if gpu_info:
    print(f"✅ GPU detected: {gpu_info}")
else:
    print("❌ No GPU found. Go to Notebook settings → Accelerator → T4 GPU or P100.")
    sys.exit(1)

internet_ok = _run("curl -s --max-time 5 -o /dev/null -w '%{http_code}' https://huggingface.co")
if internet_ok == "200":
    print("✅ Internet access confirmed.")
else:
    print("❌ No internet. Enable it in Notebook settings → Internet → On.")
    sys.exit(1)

disk = _run("df -h /kaggle/working | tail -1")
print(f"💾 Disk: {disk}")

## Cell 2 — Setup

Installs `sd-scripts` dependencies and downloads all three Anima model components (~5.6 GB total).  
All files are stored under `/kaggle/working/` which persists for the session and is saved as notebook output on commit.

**Options:**
- `hf_token` — Optional HuggingFace token for gated models. Add it as a **Kaggle Secret** named `HF_TOKEN` (Add-ons → Secrets), or paste it directly (not recommended for shared notebooks).
- `skip_download_if_exists` — Skip model download if files are already present (useful when resuming).

In [ ]:
# ── USER OPTIONS ──────────────────────────────────────────────────────────────
hf_token                = ""     # Optional. Or set via Kaggle Secret 'HF_TOKEN'
skip_download_if_exists = True   # Skip model download if files already present
# ─────────────────────────────────────────────────────────────────────────────

import os, sys, subprocess, shutil
from pathlib import Path

WORKING = Path("/kaggle/working")
SD_SCRIPTS_DIR = WORKING / "sd-scripts"
MODEL_DIR = WORKING / "anima_model"
TRAINING_DIR = WORKING / "lora_training"

for d in [MODEL_DIR, TRAINING_DIR / "configs", TRAINING_DIR / "output"]:
    d.mkdir(parents=True, exist_ok=True)

# ── Resolve HF token (Kaggle Secret preferred) ────────────────────────────────
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        print("🔑 HF_TOKEN loaded from Kaggle Secrets.")
    except Exception:
        pass  # No secret set — anonymous HF access

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

# ── Install sd-scripts ────────────────────────────────────────────────────────
print("📦 Cloning kohya-ss/sd-scripts…")
if SD_SCRIPTS_DIR.exists():
    print("   Already cloned, skipping.")
else:
    subprocess.run(
        ["git", "clone", "--depth=1", "https://github.com/kohya-ss/sd-scripts", str(SD_SCRIPTS_DIR)],
        check=True
    )

os.chdir(SD_SCRIPTS_DIR)
if str(SD_SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SD_SCRIPTS_DIR))

print("📦 Installing Python dependencies…")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "accelerate", "transformers", "diffusers", "huggingface_hub", "toml", "lion-pytorch"],
    check=True
)

# ── Configure accelerate (non-interactive) ────────────────────────────────────
accel_config = Path.home() / ".cache" / "huggingface" / "accelerate" / "default_config.yaml"
accel_config.parent.mkdir(parents=True, exist_ok=True)
accel_config.write_text("""\
compute_environment: LOCAL_MACHINE
deepspeed_config: {}
distributed_type: 'NO'
downcast_bf16: 'no'
fsdp_config: {}
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
""")
print("✅ accelerate configured.")

# ── Download Anima model components ───────────────────────────────────────────
ANIMA_REPO = "circlestone-labs/Anima"
ANIMA_FILES = [
    "anima_model.safetensors",
    "anima_vae.safetensors",
    "anima_clip.safetensors",
]

all_present = all((MODEL_DIR / f).exists() for f in ANIMA_FILES)
if skip_download_if_exists and all_present:
    print("✅ Anima model files already present, skipping download.")
else:
    from huggingface_hub import hf_hub_download
    print(f"⬇️  Downloading Anima model from {ANIMA_REPO}…")
    for fname in ANIMA_FILES:
        dest = MODEL_DIR / fname
        if dest.exists() and skip_download_if_exists:
            print(f"   {fname} already exists, skipping.")
            continue
        print(f"   Downloading {fname}…")
        hf_hub_download(
            repo_id=ANIMA_REPO,
            filename=fname,
            local_dir=str(MODEL_DIR),
            token=hf_token or None,
        )
        print(f"   ✅ {fname}")
    print("✅ All Anima model components downloaded.")

print("\n🎉 Setup complete!")
print(f"   sd-scripts : {SD_SCRIPTS_DIR}")
print(f"   model dir  : {MODEL_DIR}")
print(f"   output dir : {TRAINING_DIR / 'output'}")

## Cell 3 — (Optional) Prepare Dataset

Three ways to load your dataset:

**Option A — Kaggle Dataset (recommended for persistence)**  
Upload your dataset zip to Kaggle at https://www.kaggle.com/datasets/new, then attach it to this notebook via *Add data → Your Datasets*. It will appear under `/kaggle/input/<dataset-slug>/`.

**Option B — Unzip a file you uploaded to the session**  
Use the Kaggle file upload button (📁 icon in the top toolbar) to upload a `.zip` to `/kaggle/working/`, then set `zip_path` below.

**Option C — Use an existing folder**  
Set `image_directory` in Cell 4 directly to your folder path and skip this cell.

In [ ]:
# ── USER OPTIONS ──────────────────────────────────────────────────────────────
# Option A: path to your Kaggle Dataset input folder (read-only, will be COPIED)
kaggle_dataset_path = ""   # e.g. "/kaggle/input/my-anima-dataset/images"

# Option B: path to a .zip uploaded to /kaggle/working/
zip_path = ""              # e.g. "/kaggle/working/my_dataset.zip"

# Where to extract / copy the dataset
extract_to = "/kaggle/working/lora_training/dataset"
# ─────────────────────────────────────────────────────────────────────────────

import os, shutil, zipfile
from pathlib import Path

extract_path = Path(extract_to)
extract_path.mkdir(parents=True, exist_ok=True)

if kaggle_dataset_path and Path(kaggle_dataset_path).exists():
    src = Path(kaggle_dataset_path)
    print(f"📂 Copying from Kaggle Dataset: {src}")
    for f in src.iterdir():
        dest = extract_path / f.name
        if not dest.exists():
            shutil.copy2(f, dest)
    print(f"✅ Copied to {extract_path}")

elif zip_path and Path(zip_path).exists():
    print(f"📦 Extracting {zip_path}…")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        # Flatten single top-level folder if present
        names = zf.namelist()
        top_dirs = {n.split('/')[0] for n in names if '/' in n}
        zf.extractall(str(extract_path))
    # If everything landed in a single subfolder, flatten it
    subdirs = [d for d in extract_path.iterdir() if d.is_dir()]
    if len(subdirs) == 1 and not list(extract_path.glob("*.png")) + list(extract_path.glob("*.jpg")):
        sub = subdirs[0]
        for f in sub.iterdir():
            shutil.move(str(f), str(extract_path / f.name))
        sub.rmdir()
    print(f"✅ Extracted to {extract_path}")

else:
    print("ℹ️  No dataset source specified. Set image_directory in Cell 4 manually.")

# Count files
exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".gif"}
images = [f for f in extract_path.iterdir() if f.suffix.lower() in exts]
captions = [f for f in extract_path.iterdir() if f.suffix == ".txt"]
print(f"   Images  : {len(images)}")
print(f"   Captions: {len(captions)}")
if images and len(images) != len(captions):
    print("⚠️  Image/caption count mismatch — check your dataset!")

## Cell 4 — Training Settings

Configure your project and hyperparameters here.  
The cell will **estimate total training steps** and warn you before you start.

In [ ]:
# ── PROJECT ───────────────────────────────────────────────────────────────────
project_name     = "my_anima_lora"   # Name for output .safetensors files

# ── PATHS ─────────────────────────────────────────────────────────────────────
image_directory  = "/kaggle/working/lora_training/dataset"  # Flat dir with img+txt pairs
output_directory = "/kaggle/working/lora_training/output"   # Where .safetensors are saved

# ── TRAINING HYPERPARAMETERS ──────────────────────────────────────────────────
network_dim      = 20      # LoRA rank. Lower (e.g. 8) uses less VRAM
network_alpha    = 20      # Usually equal to network_dim
learning_rate    = 1e-4    # Reduce if you see NaN loss
max_train_epochs = 10      # Keep low to stay under session time limit
resolution       = 768     # Reduce to 512 for OOM errors
repeats          = 5       # Images × repeats = steps per epoch
batch_size       = 1       # Reduce to 1 if OOM
caption_dropout  = 0.1     # Probability of dropping a caption during training
save_every_n_epochs = 2    # Save checkpoint every N epochs (max 4 kept)
# ─────────────────────────────────────────────────────────────────────────────

import math
from pathlib import Path

img_dir = Path(image_directory)
exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".gif"}
num_images = len([f for f in img_dir.iterdir() if f.suffix.lower() in exts]) if img_dir.exists() else 0

if num_images == 0:
    print(f"❌ No images found in {image_directory}")
    print("   Check your path or re-run Cell 3.")
else:
    steps_per_epoch = math.ceil(num_images * repeats / batch_size)
    total_steps     = steps_per_epoch * max_train_epochs

    # Kaggle free tier: ~2000 steps is a safe ceiling for a T4 session
    KAGGLE_STEP_LIMIT = 2000

    print(f"📊 Step Estimate")
    print(f"   Images          : {num_images}")
    print(f"   Repeats         : {repeats}")
    print(f"   Batch size      : {batch_size}")
    print(f"   Epochs          : {max_train_epochs}")
    print(f"   Steps/epoch     : {steps_per_epoch}")
    print(f"   Total steps     : {total_steps}")
    print()

    if total_steps > KAGGLE_STEP_LIMIT:
        safe_epochs  = max(1, KAGGLE_STEP_LIMIT // steps_per_epoch)
        safe_repeats = max(1, math.floor(KAGGLE_STEP_LIMIT / (num_images / batch_size * max_train_epochs)))
        print(f"⚠️  {total_steps} steps may exceed the Kaggle session limit (~{KAGGLE_STEP_LIMIT} safe).")
        print(f"   Suggested: epochs={safe_epochs}  OR  repeats≤{safe_repeats}")
    else:
        print(f"✅ {total_steps} steps — within safe Kaggle session limit.")

Path(output_directory).mkdir(parents=True, exist_ok=True)
print(f"\n📁 Output will be saved to: {output_directory}")

## Cell 5 — Train!

Generates TOML config files and launches training via `accelerate`.  
Training output streams **line-by-line** so you can watch loss values in real time.

> 💡 **Tip:** Kaggle auto-saves everything in `/kaggle/working/` when you commit the notebook. Your `.safetensors` LoRA files will appear in the notebook's output panel.

In [ ]:
import os, sys, subprocess, toml, shutil
from pathlib import Path
from math import ceil

# ── Validate settings from Cell 4 ─────────────────────────────────────────────
assert 'project_name'     in dir() or True  # noqa — these come from Cell 4
_required = [
    'project_name', 'image_directory', 'output_directory',
    'network_dim', 'network_alpha', 'learning_rate',
    'max_train_epochs', 'resolution', 'repeats', 'batch_size',
    'caption_dropout', 'save_every_n_epochs',
]
for _v in _required:
    if _v not in vars():
        raise RuntimeError(f"Variable '{_v}' not set — please run Cell 4 first.")

WORKING       = Path("/kaggle/working")
SD_SCRIPTS    = WORKING / "sd-scripts"
MODEL_DIR     = WORKING / "anima_model"
CONFIG_DIR    = WORKING / "lora_training" / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Paths to Anima model components ───────────────────────────────────────────
pretrained_model = str(MODEL_DIR / "anima_model.safetensors")
vae_path         = str(MODEL_DIR / "anima_vae.safetensors")
clip_path        = str(MODEL_DIR / "anima_clip.safetensors")

for p in [pretrained_model, vae_path, clip_path]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Model file not found: {p} — please re-run Cell 2.")

# ── Dataset TOML ──────────────────────────────────────────────────────────────
dataset_config = {
    "general": {
        "shuffle_caption": True,
        "caption_extension": ".txt",
        "keep_tokens": 1,
    },
    "datasets": [
        {
            "resolution": resolution,
            "batch_size": batch_size,
            "subsets": [
                {
                    "image_dir": image_directory,
                    "num_repeats": repeats,
                    "caption_dropout_rate": caption_dropout,
                }
            ],
        }
    ],
}

dataset_toml = CONFIG_DIR / f"{project_name}_dataset.toml"
with open(dataset_toml, "w") as f:
    toml.dump(dataset_config, f)
print(f"✅ Dataset config: {dataset_toml}")

# ── Training TOML ─────────────────────────────────────────────────────────────
train_config = {
    "model_arguments": {
        "pretrained_model_name_or_path": pretrained_model,
        "vae": vae_path,
    },
    "additional_network_arguments": {
        "network_module": "networks.lora",
        "network_dim": network_dim,
        "network_alpha": network_alpha,
    },
    "optimizer_arguments": {
        "optimizer_type": "AdamW8bit",
        "learning_rate": learning_rate,
        "lr_scheduler": "cosine_with_restarts",
        "lr_warmup_steps": 0,
    },
    "training_arguments": {
        "output_dir": output_directory,
        "output_name": project_name,
        "save_model_as": "safetensors",
        "save_every_n_epochs": save_every_n_epochs,
        "save_last_n_epochs": 4,
        "max_train_epochs": max_train_epochs,
        "mixed_precision": "fp16",
        "gradient_checkpointing": True,
        "xformers": True,
        "cache_latents": True,
        "cache_latents_to_disk": False,
        "seed": 42,
        "clip_skip": 2,
        "max_token_length": 225,
        "noise_offset": 0.0,
    },
    "dataset_arguments": {
        "dataset_config": str(dataset_toml),
    },
    "logging_arguments": {
        "logging_dir": str(WORKING / "lora_training" / "logs"),
        "log_prefix": project_name,
    },
}

train_toml = CONFIG_DIR / f"{project_name}_train.toml"
with open(train_toml, "w") as f:
    toml.dump(train_config, f)
print(f"✅ Training config: {train_toml}")

# ── Build accelerate + train command ──────────────────────────────────────────
train_script = SD_SCRIPTS / "train_network.py"
if not train_script.exists():
    raise FileNotFoundError(f"train_network.py not found at {train_script} — re-run Cell 2.")

cmd = [
    "accelerate", "launch",
    "--num_cpu_threads_per_process=1",
    str(train_script),
    f"--config_file={train_toml}",
]

print(f"\n🚀 Starting training: {project_name}")
print(f"   Command: {' '.join(cmd)}\n")
print("=" * 60)

# ── Live-streaming output ─────────────────────────────────────────────────────
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONPATH"] = str(SD_SCRIPTS)

import sys
oom_detected = False
last_lines   = []

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
    cwd=str(SD_SCRIPTS),
    bufsize=1,
)

for line in proc.stdout:
    line = line.rstrip()
    print(line, flush=True)
    last_lines.append(line)
    if len(last_lines) > 40:
        last_lines.pop(0)
    if "out of memory" in line.lower() or "cuda oom" in line.lower():
        oom_detected = True

proc.wait()
print("=" * 60)

# ── OOM Diagnostics ───────────────────────────────────────────────────────────
if oom_detected or proc.returncode != 0:
    print("\n❌ Training ended with an error.")
    if oom_detected:
        print("💡 Out-of-Memory detected. Try:")
        print("   • Reduce resolution to 512")
        print("   • Reduce network_dim to 8")
        print("   • Set batch_size = 1")
    # dmesg may need sudo on Kaggle — attempt gracefully
    try:
        dmesg = subprocess.run(
            ["sudo", "dmesg", "--level=err,warn"],
            capture_output=True, text=True, timeout=5
        )
        if dmesg.stdout and "oom" in dmesg.stdout.lower():
            print("\n🔍 Kernel OOM messages:")
            for l in dmesg.stdout.splitlines()[-10:]:
                print(" ", l)
    except Exception:
        pass
else:
    print("\n✅ Training complete!")
    output_files = sorted(Path(output_directory).glob(f"{project_name}*.safetensors"))
    if output_files:
        print(f"\n📦 Output files ({len(output_files)}):")
        for f in output_files:
            size_mb = f.stat().st_size / 1e6
            print(f"   {f.name}  ({size_mb:.1f} MB)")
        print(f"\n💾 Files are saved to /kaggle/working/ and will be available")
        print(f"   in the notebook output panel after you commit.")
    else:
        print("⚠️  No .safetensors files found in output directory.")

## Cell 6 — (Optional) Download / Inspect Outputs

List your trained LoRA files and display their sizes.  
To download them: **Commit** the notebook (top-right button) and then go to the notebook's **Output** tab to download the `.safetensors` files directly.

In [ ]:
from pathlib import Path

# Change this if you used a different output_directory in Cell 4
output_directory = "/kaggle/working/lora_training/output"

out = Path(output_directory)
files = sorted(out.glob("*.safetensors"))

if not files:
    print("No .safetensors files found. Run Cell 5 first.")
else:
    print(f"📦 LoRA files in {out}:\n")
    for f in files:
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.name:<50}  {size_mb:6.1f} MB")
    print()
    print("To download: commit this notebook → go to Output tab → click the file.")
    print("Or use the Kaggle API:")
    print(f"  kaggle kernels output <your-username>/<notebook-slug> -p ./output")